In [ ]:
!pip install -q datasets sentence-transformers faiss-cpu google-generativeai

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd
import textwrap
import google.generativeai as genai

In [ ]:
dataset = load_dataset("lavita/MedQuAD")
dataset

In [ ]:
print(dataset)

In [ ]:
df = dataset["train"].to_pandas()

In [ ]:
df.columns

In [ ]:
df[["question", "answer", "document_source"]].isna().sum()

In [ ]:
df = df[["question", "answer", "document_source"]].dropna()
df.head()

In [ ]:
len(df)

In [ ]:
documents = []

for idx, row in df.iterrows():
    text = f"""
Question: {row['question']}

Answer: {row['answer']}
"""

    documents.append({
        "id": idx,
        "text": text,
        "source": row["document_source"],
        "question": row["question"]
    })

len(documents)

In [ ]:
print(documents[0]["text"][:1000])

In [ ]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
small_docs = documents

texts = [doc["text"] for doc in small_docs]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

embeddings = embeddings.astype("float32")

# Normalize embeddings for cosine similarity
faiss.normalize_L2(embeddings)

print("Embeddings shape:", embeddings.shape)

In [ ]:
dimension = embeddings.shape[1]

# IndexFlatIP + normalized embeddings = cosine similarity
index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Total vectors stored:", index.ntotal)

In [ ]:
import os
from langchain_groq import ChatGroq

In [ ]:
os.environ["GROQ_API_KEY"] = "Enter your API key here"


In [ ]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key=os.environ["GROQ_API_KEY"]
)

In [ ]:
def expand_query(query):
    prompt = f"""
You are a medical query expansion assistant for a RAG retriever.

Your job is to convert the user's question into a better search query.

Rules:
- Do not answer the question.
- Keep the original meaning.
- Keep common patient words and add medical synonyms.
- Prefer short keyword-style phrases, not a full sentence.
- Use both patient-friendly words and medical terms.
- Add only highly relevant medical terms.
- Do not add too many possible diseases.
- If the question is not medical, return the original question unchanged.
- Return only one line.

User Question:
{query}

Expanded Search Query:
"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

In [ ]:
def retrieve_docs(query, top_k=4, fetch_k=10, use_query_expansion=True):
    if use_query_expansion:
        search_query = expand_query(query)
    else:
        search_query = query

    query_embedding = embedding_model.encode([search_query], convert_to_numpy=True)
    query_embedding = query_embedding.astype("float32")

    # Normalize query embedding for cosine similarity
    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, fetch_k)

    results = []
    seen_questions = set()

    for score, idx in zip(scores[0], indices[0]):
        doc = small_docs[idx]

        question_key = doc["question"].strip().lower()

        if question_key in seen_questions:
            continue

        seen_questions.add(question_key)

        results.append({
            "text": doc["text"],
            "source": doc["source"],
            "question": doc["question"],
            "score": float(score),
            "expanded_query": search_query
        })

        if len(results) == top_k:
            break

    return results

In [ ]:
def is_relevant(retrieved_docs, threshold=0.35):
    if len(retrieved_docs) == 0:
        return False

    best_score = retrieved_docs[0]["score"]

    return best_score >= threshold

In [ ]:
def filter_relevant_sources(retrieved_docs, min_score=0.30):
    filtered_docs = []

    for doc in retrieved_docs:
        if doc["score"] >= min_score:
            filtered_docs.append(doc)

    return filtered_docs

In [ ]:
def generate_answer(query, retrieved_docs):
    context = "\n\n".join([
        f"[Source {i+1}]\n{doc['text']}"
        for i, doc in enumerate(retrieved_docs)
    ])

    prompt = f"""
You are a medical document assistant.

Answer the user's question using ONLY the provided sources.

Rules:
1. Use only the information from the provided sources.
2. Add citations in the answer using [Source 1], [Source 2], etc.
3. If the sources do not contain enough information, say:
   "The provided medical documents do not contain enough information."
4. Keep the answer simple and clear.
5. Do not diagnose the user.
6. Suggest consulting a healthcare professional when appropriate.

Sources:
{context}

User question:
{query}

Answer with citations:
"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

In [ ]:
def rag_chatbot(query, top_k=4, threshold=0.40, min_source_score=0.5, use_query_expansion=True):
    retrieved_docs = retrieve_docs(
        query=query,
        top_k=top_k,
        fetch_k=10,
        use_query_expansion=use_query_expansion
    )

    print("Question:")
    print(query)

    if len(retrieved_docs) > 0:
        print("\nExpanded Query:")
        print(retrieved_docs[0]["expanded_query"])

    if not is_relevant(retrieved_docs, threshold=threshold):
        print("\nAnswer:")
        print("The provided medical documents do not contain enough information.")

        print("\nReason:")
        if len(retrieved_docs) > 0:
            print(f"Best similarity score was {retrieved_docs[0]['score']}, which is below threshold {threshold}.")
        else:
            print("No documents were retrieved.")
        return

    filtered_docs = filter_relevant_sources(retrieved_docs, min_score=min_source_score)

    if len(filtered_docs) == 0:
        print("\nAnswer:")
        print("The provided medical documents do not contain enough information.")
        print("\nReason:")
        print("Retrieved documents were below the minimum source score.")
        return

    answer = generate_answer(query, filtered_docs)

    print("\nAnswer:")
    print(answer)

    print("\nRetrieved Sources:")
    for i, doc in enumerate(filtered_docs, 1):
        print(f"\n--- Source {i} ---")
        print("Dataset Source:", doc["source"])
        print("Matched Question:", doc["question"])
        print("Similarity Score:", doc["score"])

In [ ]:
rag_chatbot("What are the common symptoms of diabetes?")

In [ ]:
rag_chatbot("How is high blood pressure treated?")


In [ ]:
test_questions = [
    "What are the symptoms of diabetes?",
    "How is high blood pressure treated?",
    "What is glaucoma?",
    "I feel thirsty all the time and urinate a lot. Could this be diabetes?",
    "Why is hypertension called a silent disease?",
    "Can glaucoma cause blindness?",
    "A diabetic person skipped a meal and feels shaky, sweaty, and confused. What could be happening?",
    "What should I watch for if a baby has diarrhea and is not peeing normally?",
    "Who won the FIFA World Cup?",
    "How do I train a machine learning model?"
]

for q in test_questions:
    print("="*100)
    rag_chatbot(q)

In [ ]:
rag_chatbot("What mechanisms explain vision impairment in diabetic eye disease?")